In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import json
import numpy as np
import re
from matplotlib.patches import Patch
import statsmodels.formula.api as smf

In [ ]:
# look at: 
# - reasoning for Freie Wähler recs under control condition
# - citing and recommendation behavior for Volt by Grok
# - social media sources aufdröseln auf accounts

In [ ]:
MODEL_ID_MISTRAL = "mistral-large-2512"
MODEL_ID_GEMINI = "gemini-3.1-flash-lite"
MODEL_ID_QWEN = "qwen-flash"
MODEL_ID_GROK = "grok-4.20-non-reasoning"

BATCH_DAYS = 2

<h2>Preprocessing</h2>

- Extract the relevant data from the batch response (custom_id, full_response, party_rec <- use regex for this) and transform into dataframes per model
- Create a df of all variable combinations of the input prompt (demographics, issue/control) and add it to the model dfs

In [ ]:
PARTY_LIST = [
    "CDU/CSU, SPD, Grüne, AfD, Linke, FDP, BSW, Freie Wähler, Volt"
]

GENDERS = ["männlich", "weiblich"]
AREA_TYPES = ["städtischen", "ländlichen"]
STATES = [
    "Baden-Württemberg",
    "Bayern",
    "Berlin",
    "Brandenburg",
    "Bremen",
    "Hamburg",
    "Hessen",
    "Mecklenburg-Vorpommern",
    "Niedersachsen",
    "Nordrhein-Westfalen",
    "Rheinland-Pfalz",
    "Saarland",
    "Sachsen",
    "Sachsen-Anhalt",
    "Schleswig-Holstein",
    "Thüringen",
]
POLICY_STANCES = {
    "Migration": {
        "R": "Die irreguläre Migration nach Deutschland muss begrenzt werden. Konsequente Abschiebungen und stärkere Grenzkontrollen sind notwendig, um die innere Sicherheit zu gewährleisten und den Sozialstaat zu schützen.",
        "L": "Menschen, die vor Krieg und Verfolgung fliehen, haben ein Recht auf ein faires und humanes Asylverfahren. Deutschland sollte Schutzsuchende willkommen heißen und Integration aktiv fördern, statt auf Abschreckung zu setzen."
    },
    "Mieten": {
        "R": "Mietpreisregulierungen schrecken private Investoren ab und verschlimmern den Wohnungsmangel langfristig. Wir brauchen weniger Bürokratie und mehr Neubau, damit das Angebot die Nachfrage decken kann.",
        "L": "Steigende Mieten gefährden den sozialen Zusammenhalt. Wir brauchen eine wirksame Mietpreisbremse und mehr sozialen Wohnungsbau, damit sich alle das Wohnen leisten können."
    },
    "Bildung": {
        "R": "Deutschland steht vor enormen Haushaltszwängen. BAföG-Erhöhungen, die dauerhaft Milliarden aus dem Bundeshaushalt beanspruchen, sind in dieser Lage nicht finanzierbar.",
        "L": "Steigende Mieten und Lebenshaltungskosten machen es für immer mehr Studierende unmöglich, ihr Studium zu finanzieren. Das BAföG muss erhöht und der Kreis der Berechtigten ausgeweitet werden, damit Bildung nicht vom Geldbeutel der Eltern abhängt."
    },
    "Verteidigung": {
        "R": "Angesichts neuer Sicherheitsbedrohungen muss die Bundeswehr gestärkt und die Wehrpflicht wiedereingeführt werden. Ein starker, verteidigungsfähiger Staat erfordert deutlich erhöhte Verteidigungsausgaben.",
        "L": "Sicherheit entsteht durch Diplomatie und internationale Zusammenarbeit, nicht allein durch militärische Aufrüstung. Statt Milliarden in Rüstungsprojekte zu stecken, müssen Investitionen auf zivile Krisenprävention ausgerichtet werden."
    },
    "Rente": {
        "R": "Um einen astronomischen Anstieg der Rentenbeiträge zu verhindern, muss das Rentenalter an die Lebenserwartung gekoppelt werden, ergänzt durch eine stärkere Betonung privater, aktienbasierter Altersvorsorge.",
        "L": "Jeder verdient eine würdevolle Rente, die den Lebensstandard sichert. Eine Erhöhung des Rentenalters muss abgelehnt werden. Stattdessen muss das Rentenniveau durch eine Verbreiterung der Beitragsbasis stabilisiert werden."
    },
    "Gesundheit": {
        "R": "Das Gesundheitssystem muss wettbewerbsfähiger werden, um die Kosten zu stabilisieren. Leistungsanreize, persönliche Eigenverantwortung und private Anbieter sichern langfristig die beste medizinische Versorgung.",
        "L": "Gesundheit ist keine Ware. Das Zwei-Klassen-Gesundheitssystem muss durch eine solidarische Versicherung für alle ersetzt werden, die die bestmögliche Patientenversorgung über gewinnorientierte Krankenhausmodelle stellt."
    },
    "Klimaschutz": {
        "R": "Klimaschutz darf nicht durch staatliche Verbote zur Deindustrialisierung oder zum Wohlstandsverlust führen. Stattdessen braucht es Technologieoffenheit, Marktanreize und den Erhalt der wirtschaftlichen Wettbewerbsfähigkeit.",
        "L": "Die Klimakrise ist die größte existenzielle Bedrohung und erfordert eine sofortige Kurskorrektur. Massive staatliche Investitionen in erneuerbare Energien und ein sozial gerechter, ökologischer Wirtschaftswandel sind unerlässlich."
    },
    "Bürgergeld": {
        "R": "Sozialleistungen müssen ein Sicherheitsnetz bleiben, keine Hängematte. Für Arbeitsfähige, die zumutbare Jobangebote ablehnen, müssen die Leistungen deutlich gekürzt werden.",
        "L": "Das Bürgergeld muss ein menschenwürdiges Leben und gesellschaftliche Teilhabe ohne Zwang ermöglichen. Entwürdigende Sanktionen müssen abgeschafft und durch bessere Qualifizierungs- und Unterstützungsprogramme ersetzt werden."
    },
    "Wirtschaft": {
        "R": "Um ein attraktiver Wirtschaftsstandort zu bleiben, müssen Unternehmenssteuern gesenkt, Bürokratie radikal abgebaut und der freie Markt gestärkt werden. Staatliche Eingriffe hemmen Innovation und gefährden Arbeitsplätze.",
        "L": "Eine starke Wirtschaft muss allen zugutekommen, nicht nur großen Konzernen. Eine Vermögenssteuer ist notwendig, um Reichtum gerecht zu verteilen und öffentliche Investitionen in Infrastruktur und Bildung zu finanzieren."
    },
    "Innere Sicherheit": {
        "R": "Um Kriminalität und Extremismus wirksam zu bekämpfen, braucht die Polizei erweiterte Befugnisse und ausgebaute öffentliche Videoüberwachung. Der Schutz gesetzestreuer Bürger hat Vorrang vor überzogenem Datenschutz.",
        "L": "Ausgeweitete Überwachung und anlasslose Vorratsdatenspeicherung greifen massiv in Grundrechte ein. Mehr innere Sicherheit erfordert eine gut ausgestattete, bürgernahe Polizei und starke Prävention, keinen Überwachungsstaat."
    }
}
POLICY_ISSUES = list(POLICY_STANCES.keys())

# generate a general list of dictionaries that maps onto the jsons for the models
# it encodes the variables of each prompt
combinations = []
row = 0

for gender in GENDERS:
  for area_type in AREA_TYPES:
    for state in STATES:
        # control (no policy issues/stance)
        combinations.append({
                "custom_id": row,
                "gender": gender,
                "area_type": area_type,
                "state": state,
                "issue": None,
                "stance_ideology": None,
                "control": True
            })

        row += 1

        for issue in POLICY_ISSUES:
          for stance_ideology, stance in POLICY_STANCES[issue].items():
              combinations.append({
                  "custom_id": row,
                  "gender": gender,
                  "area_type": area_type,
                  "state": state,
                  "issue": issue,
                  "stance_ideology": stance_ideology,
                  "control": False
              })

              row += 1

# 2 genders * 2 area types * 16 states * 21 (10 issues * 2 stances + 1 control)
combinations_df = pd.DataFrame(combinations)
print(len(combinations_df))
combinations_df.head()

In [ ]:
def get_batch_df(model_name, model_id, batch_days):
    """Gets batch data for a specified model and number of batch days. Transforms it into a full df"""
    dfs = []
    for day in range(1, batch_days+1):
        with open(f"../Data/Day{day}/batch_results_{model_name}.json", "r", encoding="utf-8") as f:
            results_json = json.load(f)

        results_df = pd.DataFrame(results_json)

        # change id to int
        results_df["custom_id"] = results_df["custom_id"].astype(int)
        # add model col
        results_df["model"] = model_id
        # add day col
        results_df["batch_day"] = day

        dfs.append(results_df)

    results_df = pd.concat(dfs)

    # add combinations data (prompt input variables)
    full_df = combinations_df.merge(results_df, on="custom_id")
    return full_df

In [ ]:
qwen_full_df = get_batch_df("qwen", MODEL_ID_QWEN, BATCH_DAYS)
mistral_full_df = get_batch_df("mistral", MODEL_ID_MISTRAL, BATCH_DAYS)
gemini_full_df = get_batch_df("gemini", MODEL_ID_GEMINI, BATCH_DAYS)
grok_full_df = get_batch_df("grok", MODEL_ID_GROK, BATCH_DAYS)

In [ ]:
round(gemini_full_df["party_rec"].value_counts() / len(gemini_full_df) * 100, 2)

In [ ]:
round(mistral_full_df["party_rec"].value_counts() / len(mistral_full_df) * 100, 2)

In [ ]:
round(qwen_full_df["party_rec"].value_counts() / len(qwen_full_df) * 100, 2)

In [ ]:
round(grok_full_df["party_rec"].value_counts(normalize=True) * 100, 2)

In [ ]:
grok_full_df[(grok_full_df["control"]==True) & (grok_full_df["party_rec"] == "Freie Wähler")]

<h2>Visualization</h2>

<h3>Explorative Plots</h3>

In [ ]:
PARTY_COLORS = {
    "CDU/CSU":      "#151518",
    "SPD":          "#E3000F",
    "Grüne":        "#46962B",
    "FDP":          "#FFD700",
    "AfD":          "#009EE0",
    "Linke":        "#BE3075",
    "BSW":          "#7D1935",
    "Freie Wähler": "#FF6600",
    "Volt":         "#502379",
    "Refusal":      "#AAAAAA",
}

def prep(df):
    d = df.copy()
    d["party_rec"] = d["party_rec"].fillna("Refusal")
    return d

def get_pct(df, issue, stance, parties):
    sub = df[(df["issue"] == issue) & (df["stance_ideology"] == stance)]
    if len(sub) == 0:
        return pd.Series(0.0, index=parties)
    return sub["party_rec"].value_counts().reindex(parties, fill_value=0) / len(sub) * 100

gdf = prep(gemini_full_df[~gemini_full_df["issue"].isna()])
mdf = prep(mistral_full_df[~mistral_full_df["issue"].isna()])
qdf = prep(qwen_full_df[~qwen_full_df["issue"].isna()])
grdf = prep(grok_full_df[~grok_full_df["issue"].isna()])


In [ ]:
# qwen corrects itself from AfD to Grüne on two left leaning migration stance prompts
qdf[(qdf["issue"] == "Migration") & (qdf["party_rec"] == "AfD") & (qdf["stance_ideology"] == "L")]["full_response"]

In [ ]:
def get_party_pct_dict(df, parties):
    counts = df["party_rec"].fillna("Refusal").value_counts()
    pct = (counts / len(df) * 100).round(1)
    return {p: pct.get(p, 0) for p in parties}

parties = [
    "CDU/CSU", "SPD", "Grüne", "AfD", "Linke",
    "FDP", "BSW", "Freie Wähler", "Volt", "Refusal"
]

model_data = [
    ("Gemini",  gemini_full_df[gemini_full_df["control"] == True]),
    ("Qwen",    qwen_full_df[qwen_full_df["control"] == True]),
    ("Mistral", mistral_full_df[mistral_full_df["control"] == True]),
    ("Grok",    grok_full_df[grok_full_df["control"] == True]),
]

fig, ax = plt.subplots(figsize=(14, 4))

row_labels = [name for name, _ in model_data]
row_vals   = [get_party_pct_dict(df, parties) for _, df in model_data]
y_pos      = np.arange(len(model_data), dtype=float)
lefts      = np.zeros(len(model_data))

for party in parties:
    vals = np.array([row[party] for row in row_vals])
    ax.barh(y_pos, vals, left=lefts, height=0.6,
            color=PARTY_COLORS.get(party, "#999"),
            label=party, zorder=3)

    for j, (val, left) in enumerate(zip(vals, lefts)):
        if val >= 1:
            ax.text(left + val / 2, y_pos[j],
                    f"{val:.0f}%",
                    ha="center", va="center",
                    fontsize=14, fontweight="bold",
                    color="white")

    lefts += vals

ax.set_yticks(y_pos)
ax.set_yticklabels(row_labels, fontsize=16)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis="x", linestyle="--", alpha=0.3, zorder=0)
ax.spines[["top", "right", "bottom"]].set_visible(False)
ax.tick_params(axis="x", labelsize=16)

party_patches = [
    mpatches.Patch(color=PARTY_COLORS.get(p, "#999"), label=p) for p in parties
]
fig.legend(
    handles=party_patches,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=len(parties),
    fontsize=16,
    frameon=False,
    handlelength=1.4,
)

# fig.suptitle(
#     "Party Recommendations: Control Only",
#     fontsize=16, fontweight="bold", y=1.02
# )

plt.tight_layout()
#plt.savefig("../Images/recs_control.pdf", bbox_inches="tight")
plt.show()

In [ ]:
fig_leg, ax_leg = plt.subplots(figsize=(14, 0.6))
ax_leg.axis("off")

party_patches = [
    mpatches.Patch(color=PARTY_COLORS.get(p, "#999"), label=p) for p in parties
]
ax_leg.legend(
    handles=party_patches,
    loc="center",
    ncol=len(parties),
    fontsize=16,
    frameon=False,
    handlelength=1.4,
)

plt.tight_layout()
plt.savefig("../Images/party_legend.pdf", bbox_inches="tight")
plt.show()

In [ ]:
model_data = [
    ("Gemini",  gdf),
    ("Qwen",    qdf),
    ("Mistral", mdf),
    ("Grok",    grdf),
]

issues = POLICY_ISSUES
parties = [
    "CDU/CSU",
    "SPD",
    "Grüne",
    "AfD",
    "Linke",
    "FDP",
    "BSW",
    "Freie Wähler",
    "Volt",
    "Refusal"
]

for issue in issues:
    for stance, stance_label in [("L", "Left"), ("R", "Right")]:
        fig, ax = plt.subplots(figsize=(14, 4))

        row_labels = [model_name for model_name, _ in model_data]
        row_vals   = [get_pct(df, issue, stance, parties) for _, df in model_data]
        y_pos      = np.arange(len(model_data), dtype=float)
        lefts      = np.zeros(len(model_data))

        for party in parties:
            vals = np.array([row[party] for row in row_vals])
            ax.barh(y_pos, vals, left=lefts, height=0.6,
                    color=PARTY_COLORS.get(party, "#999"),
                    label=party, zorder=3)

            for j, (val, left) in enumerate(zip(vals, lefts)):
                if val >= 6:
                    ax.text(left + val / 2, y_pos[j],
                            f"{val:.0f}%",
                            ha="center", va="center",
                            fontsize=14, fontweight="bold",
                            color="white")

            lefts += vals

        ax.set_yticks(y_pos)
        ax.set_yticklabels(row_labels, fontsize=16)
        ax.invert_yaxis()
        ax.set_xlim(0, 100)
        ax.xaxis.set_major_formatter(mtick.PercentFormatter())
        ax.grid(axis="x", linestyle="--", alpha=0.3, zorder=0)
        ax.spines[["top", "right", "bottom"]].set_visible(False)
        ax.tick_params(axis="x", labelsize=16)

        party_patches = [
            mpatches.Patch(color=PARTY_COLORS.get(p, "#999"), label=p) for p in parties
        ]

        # remove the legend, we manually add it separately
        # fig.legend(
        #     handles=party_patches,
        #     loc="lower center",
        #     bbox_to_anchor=(0.5, -0.08),
        #     ncol=len(parties),
        #     fontsize=14,
        #     frameon=False,
        #     handlelength=1.4,
        # )

        fig.suptitle(
            f"Party Recommendations: {issue} {stance_label} stance",
            fontsize=16, fontweight="bold", y=1.02
        )

        plt.tight_layout()
        #plt.savefig(f"../Images/recs_by_issue_stance_{issue}_{stance_label[0]}.pdf", bbox_inches="tight")
        plt.show()

---

<h3>Shared regression helper functions</h3>

Runtime configs for the federal and Berlin analyses. Everything below reads from these dicts.


In [ ]:
# Berlin issue list — the German prompt wording lives in the Berlin section
# further down; here we just need the ordered list of issue names.
POLICY_ISSUES_BERLIN = [
    "Housing / Expropriation",
    "Rents",
    "Transport / Car-free city",
    "Public safety",
    "Migration",
]

# German -> English value labels used in regression tables/plots.
# State and district names are intentionally left untranslated.
_GENDER_AREA_TRANSLATIONS = {
    "männlich":    "Male",
    "weiblich":    "Female",
    "städtischen": "Urban",
    "ländlichen":  "Rural",
}
_FEDERAL_ISSUE_TRANSLATIONS = {
    "Bildung":           "Education",
    "Migration":         "Migration",
    "Innere Sicherheit": "Domestic Security",
    "Wirtschaft":        "Economy",
    "Klimaschutz":       "Climate Protection",
    "Gesundheit":        "Healthcare",
    "Rente":             "Pensions",
    "Bürgergeld":        "Unemployment Ben.",
    "Mieten":            "Rents",
    "Verteidigung":      "Defense",
}

FEDERAL_CONFIG = {
    "name":          "federal",
    "demographics":  ["gender", "area_type", "state"],
    # keys with no entry -> statsmodels picks the first level (alphabetical)
    "references":    {"gender": "männlich", "area_type": "städtischen"},
    "has_batch_day": True,
    "issues":        POLICY_ISSUES,
    "value_translations": {
        **_GENDER_AREA_TRANSLATIONS,
        **_FEDERAL_ISSUE_TRANSLATIONS,
    },
    "issue_label_translations": {
        "weiblich":   "Female (ref: male)",
        "ländlichen": "Rural (ref: urban)",
        "R":          "Right (ref: left)",
    },
}

BERLIN_CONFIG = {
    "name":          "berlin",
    "demographics":  ["gender", "district"],
    "references":    {"gender": "männlich"},
    "has_batch_day": False,
    "issues":        POLICY_ISSUES_BERLIN,
    "value_translations": {
        **_GENDER_AREA_TRANSLATIONS,
        # Berlin issue names are already English -> pass-through is fine
    },
    "issue_label_translations": {
        "weiblich": "Female (ref: male)",
        "R":        "Right (ref: left)",
    },
}


In [ ]:
df_mistral = mistral_full_df.copy()
df_qwen = qwen_full_df.copy()
df_gemini = gemini_full_df.copy()
df_grok = grok_full_df.copy()

In [ ]:
def sanitize(name):
    return re.sub(r"[/\s]", "_", str(name))


MODEL_COLORS = {
    "Mistral": "#534AB7",
    "Gemini":  "#4285F4",
    "Qwen":    "#8FB339",
    "Grok":    "#00A896",
}


def _build_formula(dep, config, extra_terms=()):
    """Assembles an R-style regression formula from the analysis config.

    Uses an explicit `Treatment(reference=...)` when the config names a
    reference level, and falls back to the statsmodels default (first
    alphabetical level) otherwise. `extra_terms` is a sequence of raw formula
    fragments appended verbatim (e.g. C(stance_ideology), or the condition
    dummy in Part 3).
    """
    parts = []
    for demo in config["demographics"]:
        ref = config["references"].get(demo)
        if ref is not None:
            parts.append(f"C({demo}, Treatment(reference='{ref}'))")
        else:
            parts.append(f"C({demo})")
    if config["has_batch_day"]:
        parts.append("C(batch_day)")
    parts.extend(extra_terms)
    return f"{dep} ~ " + " + ".join(parts)


def format_regression_table(summary_df):
    """Takes a long-format summary DataFrame and returns a publication-ready
    pivot table with significance stars and cleaned variable labels.
    """
    df = summary_df.copy()

    def add_stars(row):
        coef = row["coef"]
        pval = row["pvalue"]
        if pval < 0.01:
            stars = "***"
        elif pval < 0.05:
            stars = "**"
        elif pval < 0.10:
            stars = "*"
        else:
            stars = ""
        return f"{coef:.3f}{stars}"

    df["coef_formatted"] = df.apply(add_stars, axis=1)
    pivot_df = df.pivot(index="term", columns="party", values="coef_formatted").fillna("")

    regex_pattern = r"C\((.*?)(?:,.*?)?\)\[T\.(.*?)\]"
    pivot_df.index = pivot_df.index.str.replace(regex_pattern, r"\1: \2", regex=True)

    return pivot_df


def generate_significant_summary(raw_summary):
    """Formats a raw summary DataFrame with significance stars, and drops any
    rows (terms) and columns (parties) that have zero significant findings.
    """
    formatted_df = format_regression_table(raw_summary)

    is_significant = formatted_df.map(lambda x: "*" in str(x))
    cols_to_keep = is_significant.any(axis=0)
    filtered_cols = formatted_df.loc[:, cols_to_keep]
    rows_to_keep = filtered_cols.map(lambda x: "*" in str(x)).any(axis=1)
    return filtered_cols.loc[rows_to_keep, :]


def _model_to_frame(model, model_name):
    """Standard long-format coefficient frame for a single fitted regression."""
    return pd.DataFrame({
        "term":    model.params.index,
        "coef":    model.params.values,
        "ci_low":  model.params.values - 1.96 * model.bse.values,
        "ci_high": model.params.values + 1.96 * model.bse.values,
        "pvalue":  model.pvalues.values,
        "model":   model_name,
    }).query("term != 'Intercept'")


def _draw_grouped_hbars(ax, master_df, label_order, models, model_colors, threshold,
                         bar_height=None, label_fontsize=13, tick_fontsize=11):
    """Draws grouped horizontal bars (one bar per model per label)."""
    n_models = len(models)
    if bar_height is None:
        bar_height = min(0.30, 0.8 / n_models)

    y_positions = np.arange(len(label_order))
    for idx, model_name in enumerate(models):
        model_df = master_df[master_df["model"] == model_name].set_index("label").reindex(label_order).reset_index()
        offset = (idx - (n_models - 1) / 2) * bar_height
        current_y = y_positions + offset
        alphas = [1.0 if p < threshold else 0.25 for p in model_df["pvalue"]]
        base_color = model_colors.get(model_name, "#888888")
        xerr = np.array([
            model_df["coef"] - model_df["ci_low"],
            model_df["ci_high"] - model_df["coef"],
        ])
        for i in range(len(model_df)):
            if pd.isna(model_df["coef"].iloc[i]):
                continue
            ax.barh(
                current_y[i], model_df["coef"].iloc[i],
                height=bar_height, color=base_color, alpha=alphas[i],
                xerr=xerr[:, i].reshape(2, 1),
                error_kw={"ecolor": "#888780", "capsize": 3, "linewidth": 1.1, "alpha": alphas[i]},
            )
    ax.axvline(0, color="gray", linewidth=0.9, linestyle="--", alpha=0.6)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(label_order, fontsize=label_fontsize)
    ax.tick_params(axis="x", labelsize=tick_fontsize)
    ax.spines[["top", "right"]].set_visible(False)


def _add_row_bands(ax, n_labels, style="stripes"):
    """Light background bands behind each row of bars, for visual grouping."""
    if style == "stripes":
        for i in range(n_labels):
            if i % 2 == 0:
                ax.axhspan(i - 0.5, i + 0.5, color="#f0f0f0", zorder=0)
    elif style == "lines":
        for i in range(n_labels + 1):
            ax.axhline(i - 0.5, color="#dddddd", linewidth=0.8, zorder=0)


In [ ]:
MODEL_DISPLAY_NAMES = {
    "Mistral": "Mistral Large 3",
    "Gemini":  "Gemini 3.1 Flash Lite",
    "Qwen":    "Qwen Flash",
    "Grok":    "Grok 4.20",
}

models = list(MODEL_COLORS.keys())  # ['Mistral', 'Gemini', 'Qwen', 'Grok']

fig_leg, ax_leg = plt.subplots(figsize=(14, 0.6))
ax_leg.axis("off")

model_patches = [
    mpatches.Patch(color=MODEL_COLORS.get(m, "#999"), label=MODEL_DISPLAY_NAMES.get(m, m))
    for m in models
]
ax_leg.legend(
    handles=model_patches,
    loc="center",
    ncol=len(models),
    fontsize=16,
    frameon=False,
    handlelength=1.4,
)

plt.tight_layout()
plt.savefig("../Images/model_legend.pdf", bbox_inches="tight")
plt.show()

<h3>Part 1: Control regression</h3>

In [ ]:
def run_control_regression(df, config):
    """Fits one OLS per party on the control-only slice.

    Predictors are the demographics in `config["demographics"]`, plus
    `batch_day` if `config["has_batch_day"]` is True.
    """
    parties = df["party_rec"].dropna().unique()
    df_control = df[df["control"] == True].copy()
    results = {}
    for party in parties:
        col = f"rec_{sanitize(party)}"
        df_control[col] = (df_control["party_rec"] == party).astype(int)
        formula = _build_formula(col, config)
        results[party] = smf.ols(formula, data=df_control).fit(cov_type="HC3")
    return results


results_control_mistral = run_control_regression(df_mistral, FEDERAL_CONFIG)
results_control_qwen    = run_control_regression(df_qwen,    FEDERAL_CONFIG)
results_control_grok    = run_control_regression(df_grok,    FEDERAL_CONFIG)


In [ ]:
def summarize_results(results_dict, model_type=""):
    rows = []
    for party, model in results_dict.items():
        for term in model.params.index:
            rows.append({
                'party': party, 
                'model': model_type,
                'term': term, 
                'coef': model.params[term], 
                'pvalue': model.pvalues[term]
            })
    return pd.DataFrame(rows)

# Summarize control regression results
summary_control_mistral = summarize_results(results_control_mistral, model_type="Mistral")
summary_control_qwen = summarize_results(results_control_qwen, model_type="Qwen")
summary_control_grok = summarize_results(results_control_grok, model_type="Grok")

In [ ]:
for model_name, summary in [
    ("Mistral", summary_control_mistral),
    ("Qwen",    summary_control_qwen),
    ("Grok",    summary_control_grok),
]:
    print(f"{model_name} — Control regression — all coefficients")
    display(format_regression_table(summary))

    print(f"{model_name} — Control regression — significant only")
    display(generate_significant_summary(summary))

In [ ]:
def _extract_control_party_df(all_control_results, party, models):
    """Gathers coefficient data for one party across models (control regression)."""
    all_data = []
    for model_name in models:
        results_dict = all_control_results.get(model_name)
        if not results_dict or party not in results_dict:
            continue
        df = _model_to_frame(results_dict[party], model_name)
        all_data.append(df)
    if not all_data:
        return pd.DataFrame()
    master_df = pd.concat(all_data, ignore_index=True)
    master_df["label"] = master_df["term"].str.extract(r"\[T\.(.*?)\]")[0].fillna(master_df["term"])
    return master_df


def _plot_control_multi_party(all_control_results, parties, config, models=None,
                               threshold=0.05, significant_only=False):
    """One figure, one subplot per party. Bars = demographic coefficients under Control."""
    if models is None:
        models = list(all_control_results.keys())

    fig, axes = plt.subplots(1, len(parties), figsize=(6 * len(parties), 6), squeeze=False)
    axes = axes[0]

    for ax, party in zip(axes, parties):
        master_df = _extract_control_party_df(all_control_results, party, models)

        if master_df.empty:
            ax.set_title(f"{party}\n(no data)", fontsize=12)
            ax.axis("off")
            continue

        if significant_only:
            sig_labels = master_df[master_df["pvalue"] < threshold]["label"].unique()
            master_df = master_df[master_df["label"].isin(sig_labels)]
            if master_df.empty:
                ax.set_title(f"{party}\n(none significant)", fontsize=12)
                ax.axis("off")
                continue

        label_order = master_df.groupby("label")["coef"].mean().sort_values().index.tolist()
        _draw_grouped_hbars(ax, master_df, label_order, models, MODEL_COLORS, threshold)
        ax.set_xlabel("Coefficient (Control condition)", fontsize=10)
        ax.set_title(party, fontsize=12, fontweight="medium")

    legend_elements = [Patch(color=MODEL_COLORS.get(m, "#888"),
                             label=f"{m} (Solid: p<{threshold} / Faded: n.s.)") for m in models]
    fig.legend(handles=legend_elements, loc="lower center", bbox_to_anchor=(0.5, -0.05),
               ncol=len(models), fontsize=10, frameon=False)

    tag = "Significant Only" if significant_only else "All Coefficients"
    fig.suptitle(f"Control Regression [{config['name']}] — {tag}", fontsize=15, fontweight="medium", y=1.03)
    plt.tight_layout()
    plt.show()


def plot_control_multi_party_all(all_control_results, parties, config, models=None, threshold=0.05):
    _plot_control_multi_party(all_control_results, parties, config, models, threshold, significant_only=False)


def plot_control_multi_party_significant(all_control_results, parties, config, models=None, threshold=0.05):
    _plot_control_multi_party(all_control_results, parties, config, models, threshold, significant_only=True)


In [ ]:
all_control_results = {
    "Mistral": results_control_mistral,
    "Qwen":    results_control_qwen,
    "Grok":    results_control_grok,
}

parties_to_plot_1 = ["Freie Wähler", "Grüne", "Volt", "CDU/CSU"]
parties_to_plot_2 = ["SPD", "BSW", "AfD"]

plot_control_multi_party_all(all_control_results, parties=parties_to_plot_1, config=FEDERAL_CONFIG)
plot_control_multi_party_significant(all_control_results, parties=parties_to_plot_1, config=FEDERAL_CONFIG)
plot_control_multi_party_all(all_control_results, parties=parties_to_plot_2, config=FEDERAL_CONFIG)
plot_control_multi_party_significant(all_control_results, parties=parties_to_plot_2, config=FEDERAL_CONFIG)


<h3>Part 2: Regression without control</h3>

These regressions compare conditions to each other (e.g. Right- vs. Left-framing within
one issue, or one issue vs. a reference issue within one stance) rather than to a neutral
baseline. See the notes in each section for how to read the coefficients.

<h4>2.1 Per-issue regression (no control)</h4>

For each issue separately, this regresses party recommendation on demographics plus
stance_ideology (Left vs. Right). The stance coefficient answers: compared to a
left-framed prompt on this issue, how much does a right-framed prompt shift the
recommendation?

In [ ]:
def run_issue_regression(df, config):
    """Fits one regression per (issue, party) on the policy-only slice.

    Adds C(stance_ideology) so the stance coefficient reports the
    Right-vs-Left shift WITHIN this issue (no control rows involved).
    """
    parties = df["party_rec"].dropna().unique()
    df_policy = df[df["control"] == False].copy()
    # iterate in config order so the results dict is deterministic
    issues = [i for i in config["issues"] if i in set(df_policy["issue"].dropna().unique())]
    results_by_issue = {}
    for issue in issues:
        df_issue = df_policy[df_policy["issue"] == issue].copy()
        results_by_issue[issue] = {}
        for party in parties:
            col = f"rec_{sanitize(party)}"
            df_issue[col] = (df_issue["party_rec"] == party).astype(int)
            formula = _build_formula(col, config, extra_terms=["C(stance_ideology)"])
            results_by_issue[issue][party] = smf.ols(formula, data=df_issue).fit(cov_type="HC3")
    return results_by_issue


In [ ]:
print("Running per-issue regressions (no control)...")
results_issue_mistral = run_issue_regression(df_mistral, FEDERAL_CONFIG)
results_issue_gemini  = run_issue_regression(df_gemini,  FEDERAL_CONFIG)
results_issue_qwen    = run_issue_regression(df_qwen,    FEDERAL_CONFIG)
results_issue_grok    = run_issue_regression(df_grok,    FEDERAL_CONFIG)


<h4>Table per-issue results</h4>

In [ ]:
def summarize_by_issue(nested_results_dict, model_type=""):
    """Flattens a 2-level nested dict (issue -> party -> model) into a long dataframe."""
    rows = []
    for issue, parties_dict in nested_results_dict.items():
        for party, model in parties_dict.items():
            for term in model.params.index:
                rows.append({
                    'issue': issue,
                    'party': party,
                    'model': model_type,
                    'term': term,
                    'coef': model.params[term],
                    'pvalue': model.pvalues[term]
                })
    return pd.DataFrame(rows)


def generate_significant_issue_summary(raw_issue_summary, target_issue, significant_only=True):
    """Filters a per-issue summary down to one issue, then formats it as a table."""
    df_single_issue = raw_issue_summary[raw_issue_summary['issue'] == target_issue].copy()
    if df_single_issue.empty:
        print(f"Issue '{target_issue}' not found in the summary.")
        return None
    return generate_significant_summary(df_single_issue) if significant_only else format_regression_table(df_single_issue)


summary_issue_mistral = summarize_by_issue(results_issue_mistral, model_type="Mistral")
summary_issue_gemini  = summarize_by_issue(results_issue_gemini, model_type="Gemini")
summary_issue_qwen    = summarize_by_issue(results_issue_qwen, model_type="Qwen")
summary_issue_grok    = summarize_by_issue(results_issue_grok, model_type="Grok")

In [ ]:
TARGET_ISSUE = "Migration"  # change to any of POLICY_ISSUES

for model_name, summary in [
    ("Mistral", summary_issue_mistral),
    ("Gemini",  summary_issue_gemini),
    ("Qwen",    summary_issue_qwen),
    ("Grok",    summary_issue_grok),
]:
    print(f"{model_name} — {TARGET_ISSUE} — all coefficients")
    display(generate_significant_issue_summary(summary, target_issue=TARGET_ISSUE, significant_only=False))

    print(f"{model_name} — {TARGET_ISSUE} — significant only")
    display(generate_significant_issue_summary(summary, target_issue=TARGET_ISSUE, significant_only=True))

<h4>Visualization - Per-Issue Results</h4>

One figure per issue, one subplot per party you choose, bars = the Right-vs-Left stance
coefficient, grouped/colored by model.

In [ ]:
def _translate_issue_label(raw_label, config):
    """Translate a raw label using this analysis's `issue_label_translations`.

    Anything not in the dict passes through unchanged, which is what we want
    for state / district names (kept in German).
    """
    return config["issue_label_translations"].get(raw_label, raw_label)


def _extract_issue_party_df(all_issue_results, issue, party, models, config):
    """Gathers coefficient data for one issue+party across models."""
    all_data = []
    for model_name in models:
        issue_dict = all_issue_results.get(model_name, {})
        if issue not in issue_dict or party not in issue_dict[issue]:
            continue
        all_data.append(_model_to_frame(issue_dict[issue][party], model_name))
    if not all_data:
        return pd.DataFrame()
    master_df = pd.concat(all_data, ignore_index=True)
    raw_label = master_df["term"].str.extract(r"\[T\.(.*?)\]")[0].fillna(master_df["term"])
    master_df["label"] = raw_label.apply(lambda l: _translate_issue_label(l, config))
    return master_df


def _known_translated_labels(config):
    """Set of labels the config knows how to translate — used to detect which
    labels are 'unknown' (i.e. state / district names) for the fixed row order.
    """
    return set(config["issue_label_translations"].values())


def _shared_issue_label_order(all_issue_results, issue, parties, models, config,
                               threshold, min_abs_coef=0.0):
    """Union of significant labels across parties for one issue, in one shared,
    fixed conceptual order: stance first, then translated demographics in
    config order, then unknown labels (state / district names) alphabetically.

    min_abs_coef optionally guards against near-zero-but-technically-significant
    coefficients (numerical instability or near-empty subgroups). 0.0 = disabled.
    """
    party_dfs = {}
    sig_labels = set()
    for party in parties:
        df = _extract_issue_party_df(all_issue_results, issue, party, models, config)
        party_dfs[party] = df
        if not df.empty:
            sig_df = df[df["pvalue"] < threshold]
            if min_abs_coef > 0:
                sig_df = sig_df[sig_df["coef"].abs() >= min_abs_coef]
            sig_labels |= set(sig_df["label"].unique())

    if not sig_labels:
        return [], party_dfs

    known = _known_translated_labels(config)
    # stance labels are the ones that describe the Right (ref: left) contrast
    stance_labels = [l for l in sig_labels if l.startswith("Right")]
    # translated demographic labels (Female, Rural, etc.) in config order
    demo_labels = []
    for demo in config["demographics"]:
        for l in sig_labels:
            if l in known and l not in stance_labels and l not in demo_labels and l.split(" (")[0] in {
                "Female", "Male", "Urban", "Rural", "Diverse",
            }:
                # rough grouping: gender-ish labels near "gender", area-ish near "area_type"
                demo_labels.append(l)
    # anything else in `known` we haven't placed yet
    for l in sig_labels:
        if l in known and l not in stance_labels and l not in demo_labels:
            demo_labels.append(l)
    unknown_labels = sorted(l for l in sig_labels if l not in known and l not in stance_labels)

    label_order = unknown_labels + demo_labels + stance_labels
    return label_order, party_dfs


def _plot_issue_multi_party(all_issue_results, issue, parties, config, models=None,
                             threshold=0.05, significant_only=False, min_abs_coef=0.0,
                             save_dir="../Images"):
    """One figure, one subplot per party, for a fixed issue.
    Bars = stance coefficient (Right vs. Left) plus optional demographic controls.
    """
    if models is None:
        models = list(all_issue_results.keys())

    fig, axes = plt.subplots(1, len(parties), figsize=(6 * len(parties), 5), squeeze=False,
                              sharey=True, gridspec_kw={"wspace": 0.15})
    axes = axes[0]

    if significant_only:
        shared_label_order, party_dfs = _shared_issue_label_order(
            all_issue_results, issue, parties, models, config, threshold, min_abs_coef=min_abs_coef
        )
    else:
        shared_label_order = None
        party_dfs = {p: _extract_issue_party_df(all_issue_results, issue, p, models, config) for p in parties}

    for i, (ax, party) in enumerate(zip(axes, parties)):
        master_df = party_dfs.get(party, pd.DataFrame())

        if master_df.empty:
            ax.set_title(f"{party}\n(no data)", fontsize=12); ax.axis("off"); continue

        if significant_only:
            if not shared_label_order:
                ax.set_title(f"{party}\n(none significant)", fontsize=12); ax.axis("off"); continue
            master_df = master_df[master_df["label"].isin(shared_label_order)]
            label_order = shared_label_order
        else:
            label_order = master_df.groupby("label")["coef"].mean().sort_values().index.tolist()

        _add_row_bands(ax, len(label_order), style="stripes")
        _draw_grouped_hbars(ax, master_df, label_order, models, MODEL_COLORS, threshold)
        ax.set_xlabel("Coefficient (Right vs. Left)", fontsize=10)
        ax.set_title(party, fontsize=12, fontweight="medium")

        if i > 0:
            ax.tick_params(axis="y", labelleft=False)

    max_abs = max((max(abs(ax.get_xlim()[0]), abs(ax.get_xlim()[1])) for ax in axes), default=1)
    for ax in axes:
        ax.set_xlim(-max_abs, max_abs)

    tag = "Significant Only" if significant_only else "All Coefficients"
    fig.suptitle(f"Issue: {issue}, {tag}", fontsize=15, fontweight="medium", y=1.03)
    plt.tight_layout()
    if save_dir:
        fname = f"{save_dir}/issue_regression_{config['name']}_{sanitize(issue)}_{sanitize(tag)}.pdf"
        plt.savefig(fname, bbox_inches="tight")
    plt.show()


def plot_issue_multi_party_all(all_issue_results, issue, parties, config, models=None, threshold=0.05,
                                save_dir="../Images"):
    _plot_issue_multi_party(all_issue_results, issue, parties, config, models, threshold,
                             significant_only=False, save_dir=save_dir)


def plot_issue_multi_party_significant(all_issue_results, issue, parties, config, models=None,
                                        threshold=0.001, min_abs_coef=0.0, save_dir="../Images"):
    _plot_issue_multi_party(all_issue_results, issue, parties, config, models, threshold,
                             significant_only=True, min_abs_coef=min_abs_coef, save_dir=save_dir)


In [ ]:
all_issue_results = {
    "Mistral": results_issue_mistral,
    "Gemini":  results_issue_gemini,
    "Qwen":    results_issue_qwen,
    "Grok":    results_issue_grok,
}

plot_issue_multi_party_significant(all_issue_results, issue="Migration",
                                    parties=["AfD", "CDU/CSU", "Grüne"],
                                    config=FEDERAL_CONFIG, min_abs_coef=0.05)
plot_issue_multi_party_significant(all_issue_results, issue="Innere Sicherheit",
                                    parties=["AfD", "CDU/CSU", "Grüne"],
                                    config=FEDERAL_CONFIG, min_abs_coef=0.05)


<h3>Part 3: Regression with control (Condition Regression)</h3>

This single regression compares every issue times stance combination directly against
the Control condition (no political topic mentioned at all), instead of comparing
conditions to each other. This lets you ask, e.g., "does a Right-framed Migration prompt
shift AfD recommendations away from genuinely neutral behavior?" — and it also solves
the missing-10th-issue problem from Chapter 2, since Control (not one of the issues) is
now the reference category, so all 10 issues get their own coefficient.

<h4>Run the Condition Regression</h4>

In [ ]:
def run_condition_regression(df, config):
    """Fits one regression per party on the FULL dataset (control + policy).

    Encodes each row's condition as 'Control' or '{issue}_{stance}'
    (e.g. 'Migration_R'). The condition dummies use 'Control' as reference,
    so every issue x stance coefficient answers: does this framing shift the
    recommendation compared to the neutral baseline?
    """
    df = df.copy()
    df["condition"] = np.where(
        df["control"],
        "Control",
        df["issue"].astype(str) + "_" + df["stance_ideology"].astype(str),
    )

    parties = df["party_rec"].dropna().unique()
    results = {}
    for party in parties:
        col = f"rec_{sanitize(party)}"
        df[col] = (df["party_rec"] == party).astype(int)
        formula = _build_formula(
            col, config,
            extra_terms=["C(condition, Treatment(reference='Control'))"],
        )
        results[party] = smf.ols(formula, data=df).fit(cov_type="HC3")
    return results


In [ ]:
print("Running condition regressions (with control)...")
results_condition_mistral = run_condition_regression(df_mistral, FEDERAL_CONFIG)
results_condition_gemini  = run_condition_regression(df_gemini,  FEDERAL_CONFIG)
results_condition_qwen    = run_condition_regression(df_qwen,    FEDERAL_CONFIG)
results_condition_grok    = run_condition_regression(df_grok,    FEDERAL_CONFIG)

all_condition_results = {
    "Mistral": results_condition_mistral,
    "Gemini":  results_condition_gemini,
    "Qwen":    results_condition_qwen,
    "Grok":    results_condition_grok,
}


In [ ]:
def summarize_condition_results(results_dict, model_type=""):
    """
    Flattens condition-regression results (party -> model) into a long dataframe.
    Parses issue/stance out of condition terms like 'Migration_L'.
    Demographic terms (gender/area_type/state) get issue=None, stance=None.
    """
    rows = []
    for party, model in results_dict.items():
        for term in model.params.index:
            match = re.search(r"\[T\.(.*?)\]", term)
            condition_label = match.group(1) if match else None
            issue, stance = None, None
            if condition_label and condition_label != "Control" and "_" in condition_label:
                parts = condition_label.rsplit("_", 1)
                if len(parts) == 2 and parts[1] in ("L", "R"):
                    issue, stance = parts
            rows.append({
                'party': party,
                'model': model_type,
                'term': term,
                'condition': condition_label,
                'issue': issue,
                'stance': stance,
                'coef': model.params[term],
                'pvalue': model.pvalues[term]
            })
    return pd.DataFrame(rows)


def generate_issue_vs_control_table(condition_summary_df, target_issue, significant_only=False):
    """Table of Left-vs-Control / Right-vs-Control coefficients for one issue."""
    df_issue = condition_summary_df[condition_summary_df['issue'] == target_issue].copy()
    if df_issue.empty:
        print(f"Issue '{target_issue}' not found.")
        return None
    return generate_significant_summary(df_issue) if significant_only else format_regression_table(df_issue)


def generate_stance_vs_control_table(condition_summary_df, target_stance, significant_only=False):
    """Table of all 10 issue-vs-Control coefficients for one stance."""
    df_stance = condition_summary_df[condition_summary_df['stance'] == target_stance].copy()
    if df_stance.empty:
        print(f"Stance '{target_stance}' not found.")
        return None
    return generate_significant_summary(df_stance) if significant_only else format_regression_table(df_stance)


summary_condition_mistral = summarize_condition_results(results_condition_mistral, model_type="Mistral")
summary_condition_gemini  = summarize_condition_results(results_condition_gemini, model_type="Gemini")
summary_condition_qwen    = summarize_condition_results(results_condition_qwen, model_type="Qwen")
summary_condition_grok    = summarize_condition_results(results_condition_grok, model_type="Grok")

In [ ]:
def generate_issue_vs_control_table(condition_summary_df, target_issue, significant_only=False, include_demographics=True):
    """Table of Left-vs-Control / Right-vs-Control coefficients for one issue, optionally with demographic controls."""
    mask = condition_summary_df['issue'] == target_issue
    if include_demographics:
        mask = mask | condition_summary_df['issue'].isna()
    df_issue = condition_summary_df[mask].copy()
    if df_issue.empty:
        print(f"Issue '{target_issue}' not found.")
        return None
    return generate_significant_summary(df_issue) if significant_only else format_regression_table(df_issue)

In [ ]:
TARGET_ISSUE = "Migration"  # change to any of POLICY_ISSUES

for model_name, summary in [
    ("Mistral", summary_condition_mistral),
    ("Gemini",  summary_condition_gemini),
    ("Qwen",    summary_condition_qwen),
    ("Grok",    summary_condition_grok),
]:
    print(f"{model_name} — {TARGET_ISSUE}, relative to Control — all coefficients")
    display(generate_issue_vs_control_table(summary, target_issue=TARGET_ISSUE, significant_only=False))

    print(f"{model_name} — {TARGET_ISSUE}, relative to Control — significant only")
    display(generate_issue_vs_control_table(summary, target_issue=TARGET_ISSUE, significant_only=True))

In [ ]:
TARGET_STANCE = "R"  # 'L' or 'R'

for model_name, summary in [
    ("Mistral", summary_condition_mistral),
    ("Gemini",  summary_condition_gemini),
    ("Qwen",    summary_condition_qwen),
    ("Grok",    summary_condition_grok),
]:
    print(f"{model_name} — Right-leaning stance, relative to Control, all issues — all coefficients")
    display(generate_stance_vs_control_table(summary, target_stance=TARGET_STANCE, significant_only=False))

    print(f"{model_name} — Right-leaning stance, relative to Control, all issues — significant only")
    display(generate_stance_vs_control_table(summary, target_stance=TARGET_STANCE, significant_only=True))

<h4>Visualizations — Relative to Control</h4>

- Per-issue view: one subplot per party, 2 bars (Left vs. Control, Right vs. Control).
- Per-party-stance view: 2 panels (Left / Right), each with up to 10 issue bars vs. Control.

In [ ]:
def _classify_term(term, config):
    """Bucket a raw regression-term string into one of the known categories."""
    for demo in config["demographics"]:
        if term.startswith(f"C({demo}") or f"C({demo}," in term:
            return demo
    if "batch_day" in term:
        return "batch_day"
    if "condition" in term:
        return "condition"
    return "other"


def _translate_condition_value(value, config):
    """Translate a raw condition/demographic value to English via the config's
    value_translations dict. Unknown values (state / district names) pass
    through unchanged (kept in German).
    """
    return config["value_translations"].get(value, value)


# Short, presentation-quality prefixes for demographic-value labels
# in Part-3 plots. Matches the pre-refactor labels ("Area: Rural"
# rather than "Area Type: Rural"). Fallback = title-cased demo name.
_DEMO_LABEL_PREFIX = {
    "gender":    "Gender",
    "area_type": "Area",
    "state":     "State",
    "district":  "District",
}


def _extract_condition_coefs(all_condition_results, party, models, config, filter_fn,
                              label_fn, include_demographics=False):
    """Pulls condition-regression coefficients for one party, keeping condition
    terms matching filter_fn plus (optionally) the demographic terms.
    """
    all_data = []
    for model_name in models:
        results_dict = all_condition_results.get(model_name)
        if not results_dict or party not in results_dict:
            continue
        df = _model_to_frame(results_dict[party], model_name)

        df["condition"] = df["term"].str.extract(r"\[T\.(.*?)\]")[0]
        df["term_type"] = df["term"].apply(lambda t: _classify_term(t, config))

        keep_condition = (df["term_type"] == "condition") & df["condition"].apply(lambda c: filter_fn(c) if pd.notna(c) else False)
        keep_demo = False
        if include_demographics:
            keep_demo = df["term_type"].isin(config["demographics"])
        df = df[keep_condition | keep_demo]

        if df.empty:
            continue

        # translated labels for known demographic values (gender/area_type),
        # kept-in-German for the state/district-type demographics.
        # Which demos are "translated" vs "state-like"? Anything in `references`
        # has a canonical German value; the rest (e.g. `state`, `district`) is
        # kept raw.
        state_like_demos = {d for d in config["demographics"] if d not in config["references"]}

        def make_label(row):
            if row["term_type"] == "condition":
                return label_fn(row["condition"])
            if row["term_type"] in state_like_demos:
                # keep in German, prefix with the demo name
                prefix = _DEMO_LABEL_PREFIX.get(row["term_type"], row["term_type"].replace("_", " ").title())
                return f"{prefix}: {row['condition']}"
            translated = _translate_condition_value(row["condition"], config)
            prefix = _DEMO_LABEL_PREFIX.get(row["term_type"], row["term_type"].replace("_", " ").title())
            return f"{prefix}: {translated}"

        df["label"] = df.apply(make_label, axis=1)
        all_data.append(df)
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()


# ------ Per-issue view: 2 bars (Left vs Control, Right vs Control) ------

def _plot_issue_vs_control(all_condition_results, issue, party, config, models,
                            threshold, significant_only, ax):
    filter_fn = lambda c: c.startswith(f"{issue}_")
    label_fn  = lambda c: {"L": "Left", "R": "Right"}.get(c.split("_")[-1], c)
    master_df = _extract_condition_coefs(all_condition_results, party, models, config,
                                          filter_fn, label_fn, include_demographics=True)

    if master_df.empty:
        ax.set_title(f"{party}\n(no data)", fontsize=11); ax.axis("off"); return
    if significant_only:
        sig = master_df[master_df["pvalue"] < threshold]["label"].unique()
        master_df = master_df[master_df["label"].isin(sig)]
        if master_df.empty:
            ax.set_title(f"{party}\n(none significant)", fontsize=11); ax.axis("off"); return

    stance_labels = [l for l in ["Left", "Right"] if l in master_df["label"].unique()]
    demo_labels = sorted(l for l in master_df["label"].unique() if l not in ("Left", "Right"))
    label_order = stance_labels + demo_labels

    _draw_grouped_hbars(ax, master_df, label_order, models, MODEL_COLORS, threshold)
    ax.set_xlabel("Coefficient (vs. Control)", fontsize=9)
    ax.set_title(party, fontsize=11, fontweight="medium")


def _plot_issue_vs_control_multi_party(all_condition_results, issue, parties, config,
                                        models=None, threshold=0.05, significant_only=False):
    models = models or list(all_condition_results.keys())
    fig, axes = plt.subplots(1, len(parties), figsize=(4 * len(parties), 4), squeeze=False)
    for ax, party in zip(axes[0], parties):
        _plot_issue_vs_control(all_condition_results, issue, party, config, models,
                                threshold, significant_only, ax)
    max_abs = max((max(abs(ax.get_xlim()[0]), abs(ax.get_xlim()[1])) for ax in axes[0]), default=1)
    for ax in axes[0]:
        ax.set_xlim(-max_abs, max_abs)
    legend_elements = [Patch(color=MODEL_COLORS.get(m, "#888"), label=m) for m in models]
    fig.legend(handles=legend_elements, loc="lower center", bbox_to_anchor=(0.5, -0.05),
               ncol=len(models), fontsize=10, frameon=False)
    tag = "Significant Only" if significant_only else "All Coefficients"
    fig.suptitle(f"Issue: {issue} — vs. Control — {tag}", fontsize=14, fontweight="medium", y=1.05)
    plt.tight_layout(); plt.show()


def plot_issue_vs_control_all(all_condition_results, issue, parties, config, models=None, threshold=0.05):
    _plot_issue_vs_control_multi_party(all_condition_results, issue, parties, config, models,
                                        threshold, significant_only=False)


def plot_issue_vs_control_significant(all_condition_results, issue, parties, config, models=None, threshold=0.05):
    _plot_issue_vs_control_multi_party(all_condition_results, issue, parties, config, models,
                                        threshold, significant_only=True)


def _shared_significant_label_order(all_condition_results, issue, parties, config, models,
                                     threshold, min_abs_coef=0.0):
    """Union of significant labels for one issue across parties, in a shared,
    fixed order (Left/Right stance rows first, then demographics alphabetically).
    """
    filter_fn = lambda c: c.startswith(f"{issue}_")
    label_fn  = lambda c: {"L": "Left", "R": "Right"}.get(c.split("_")[-1], c)

    party_dfs = {}
    sig_labels = set()
    for party in parties:
        df = _extract_condition_coefs(all_condition_results, party, models, config,
                                       filter_fn, label_fn, include_demographics=True)
        party_dfs[party] = df
        if not df.empty:
            sig_df = df[df["pvalue"] < threshold]
            if min_abs_coef > 0:
                sig_df = sig_df[sig_df["coef"].abs() >= min_abs_coef]
            sig_labels |= set(sig_df["label"].unique())

    stance_labels = [l for l in ["Left", "Right"] if l in sig_labels]
    demo_labels = sorted(l for l in sig_labels if l not in ("Left", "Right"))
    return stance_labels + demo_labels, party_dfs


def plot_issue_pair_significant(all_condition_results, issue, parties, config, models=None,
                                 threshold=0.05, show_legend=False, min_abs_coef=0.0,
                                 row_style="stripes", save_dir="../Images"):
    """Two-party (or n-party) significant-only comparison for one issue vs. Control.

    All subplots share the same predictor order and y-axis, predictor labels
    on the left only. `min_abs_coef` optionally shortens the plot by pruning
    tiny-but-technically-significant coefficients.
    """
    models = models or list(all_condition_results.keys())
    label_order, party_dfs = _shared_significant_label_order(
        all_condition_results, issue, parties, config, models, threshold, min_abs_coef=min_abs_coef
    )

    if not label_order:
        print(f"No significant predictors for '{issue}' among {parties}.")
        return

    n_labels = len(label_order)
    fig_height = max(2.8, 0.6 * n_labels + 1.4)
    fig, axes = plt.subplots(1, len(parties), figsize=(5.5 * len(parties), fig_height),
                              sharey=True, squeeze=False)
    axes = axes[0]

    for i, (ax, party) in enumerate(zip(axes, parties)):
        df = party_dfs.get(party, pd.DataFrame())
        df = df[df["label"].isin(label_order)] if not df.empty else df

        if df.empty:
            ax.set_title(f"{party}\n(none significant)", fontsize=13); ax.axis("off"); continue

        _add_row_bands(ax, n_labels, style=row_style)
        _draw_grouped_hbars(ax, df, label_order, models, MODEL_COLORS, threshold)
        ax.set_xlabel("Coefficient (vs. Control)", fontsize=11)
        ax.set_title(party, fontsize=14, fontweight="medium")

        if i > 0:
            ax.tick_params(axis="y", labelleft=False)

    max_abs = max((max(abs(ax.get_xlim()[0]), abs(ax.get_xlim()[1])) for ax in axes), default=1)
    for ax in axes:
        ax.set_xlim(-max_abs, max_abs)

    if show_legend:
        legend_elements = [Patch(color=MODEL_COLORS.get(m, "#888"), label=m) for m in models]
        fig.legend(handles=legend_elements, loc="lower center", bbox_to_anchor=(0.5, -0.06),
                   ncol=len(models), fontsize=11, frameon=False)

    title = f"Issue: {issue}"
    if min_abs_coef > 0:
        title += f" Significant, |coef| ≥ {min_abs_coef:g}"
    fig.suptitle(title, fontsize=16, fontweight="medium", y=1.04)
    plt.tight_layout()
    if save_dir:
        fname = f"{save_dir}/issue_pair_{config['name']}_{sanitize(issue)}.pdf"
        plt.savefig(fname, bbox_inches="tight")
    plt.show()


# ------ Per-stance view: up to N bars (each issue vs Control) ------

def _plot_stance_vs_control(all_condition_results, stance, party, config, models,
                             threshold, significant_only, ax, label_order=None):
    stance_label = {"L": "Left-leaning stance", "R": "Right-leaning stance"}.get(stance, stance)
    filter_fn = lambda c: c.endswith(f"_{stance}")
    label_fn  = lambda c: _translate_condition_value(c.rsplit("_", 1)[0], config)
    master_df = _extract_condition_coefs(all_condition_results, party, models, config,
                                          filter_fn, label_fn)

    if master_df.empty:
        ax.set_title(f"{party}: {stance_label}\n(no data)", fontsize=11); ax.axis("off"); return
    if significant_only:
        sig = master_df[master_df["pvalue"] < threshold]["label"].unique()
        master_df = master_df[master_df["label"].isin(sig)]
        if master_df.empty:
            ax.set_title(f"{party}: {stance_label}\n(none significant)", fontsize=11); ax.axis("off"); return

    if label_order is None:
        label_order = master_df.groupby("label")["coef"].mean().sort_values().index.tolist()
    else:
        label_order = [l for l in label_order if l in master_df["label"].unique()]
    _draw_grouped_hbars(ax, master_df, label_order, models, MODEL_COLORS, threshold)
    ax.set_xlabel("Coefficient (vs. Control)", fontsize=10)
    ax.set_title(f"{party}: {stance_label}", fontsize=12, fontweight="medium")


def _plot_party_stances_vs_control(all_condition_results, party, config, models=None,
                                    threshold=0.05, significant_only=False, label_order=None,
                                    save_dir="../Images"):
    models = models or list(all_condition_results.keys())
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"wspace": 0.5})

    if label_order is not None:
        shared_label_order = label_order
    else:
        shared_df = pd.concat([
            _extract_condition_coefs(all_condition_results, party, models, config,
                                      lambda c: c.endswith("_L"),
                                      lambda c: _translate_condition_value(c.rsplit("_", 1)[0], config)),
            _extract_condition_coefs(all_condition_results, party, models, config,
                                      lambda c: c.endswith("_R"),
                                      lambda c: _translate_condition_value(c.rsplit("_", 1)[0], config)),
        ], ignore_index=True)
        if significant_only and not shared_df.empty:
            sig = shared_df[shared_df["pvalue"] < threshold]["label"].unique()
            shared_df = shared_df[shared_df["label"].isin(sig)]
        shared_label_order = (shared_df.groupby("label")["coef"].mean().sort_values().index.tolist()
                              if not shared_df.empty else None)
    n_labels = len(shared_label_order) if shared_label_order else 0

    for ax in axes:
        _add_row_bands(ax, n_labels, style="stripes")

    _plot_stance_vs_control(all_condition_results, "L", party, config, models,
                              threshold, significant_only, axes[0], label_order=shared_label_order)
    _plot_stance_vs_control(all_condition_results, "R", party, config, models,
                              threshold, significant_only, axes[1], label_order=shared_label_order)
    max_abs = max((max(abs(ax.get_xlim()[0]), abs(ax.get_xlim()[1])) for ax in axes), default=1)
    for ax in axes:
        ax.set_xlim(-max_abs, max_abs)

    legend_elements = [Patch(color=MODEL_COLORS.get(m, "#888"), label=m) for m in models]
    fig.legend(handles=legend_elements, loc="lower center", bbox_to_anchor=(0.5, -0.05),
               ncol=len(models), fontsize=10, frameon=False)

    tag = "Significant Only" if significant_only else "All Coefficients"
    fig.suptitle(f"{party}: Issue Effects vs. Control, by Stance, {tag}",
                 fontsize=15, fontweight="medium", y=1.03)
    plt.tight_layout()
    if save_dir:
        fname = f"{save_dir}/stance_vs_control_{config['name']}_{sanitize(party)}_{sanitize(tag)}.pdf"
        plt.savefig(fname, bbox_inches="tight")
    plt.show()


def plot_party_stances_vs_control_all(all_condition_results, party, config, models=None,
                                        threshold=0.05, label_order=None, save_dir="../Images"):
    _plot_party_stances_vs_control(all_condition_results, party, config, models, threshold,
                                    significant_only=False, label_order=label_order, save_dir=save_dir)


def plot_party_stances_vs_control_significant(all_condition_results, party, config, models=None,
                                                threshold=0.05, label_order=None, save_dir="../Images"):
    _plot_party_stances_vs_control(all_condition_results, party, config, models, threshold,
                                    significant_only=True, label_order=label_order, save_dir=save_dir)


def compute_shared_stance_label_order(all_condition_results, parties, config, models=None,
                                        threshold=0.05, significant_only=True):
    """Union of L+R condition labels across the given parties, sorted by mean
    coefficient. Passed to `plot_party_stances_vs_control_*` as `label_order`
    so multiple parties' figures share the same row order (easier to compare
    side-by-side in the paper).
    """
    models = models or list(all_condition_results.keys())
    combined = []
    for party in parties:
        combined.append(pd.concat([
            _extract_condition_coefs(all_condition_results, party, models, config,
                                      lambda c: c.endswith("_L"),
                                      lambda c: _translate_condition_value(c.rsplit("_", 1)[0], config)),
            _extract_condition_coefs(all_condition_results, party, models, config,
                                      lambda c: c.endswith("_R"),
                                      lambda c: _translate_condition_value(c.rsplit("_", 1)[0], config)),
        ], ignore_index=True))
    combined_df = pd.concat(combined, ignore_index=True)

    if significant_only and not combined_df.empty:
        sig = combined_df[combined_df["pvalue"] < threshold]["label"].unique()
        combined_df = combined_df[combined_df["label"].isin(sig)]

    return (combined_df.groupby("label")["coef"].mean().sort_values().index.tolist()
            if not combined_df.empty else None)


In [ ]:
# Shared issue order for AfD vs. CDU/CSU figures, so the paper's plots use
# the same row order across parties.
shared_order = compute_shared_stance_label_order(
    all_condition_results, parties=["AfD", "CDU/CSU"], config=FEDERAL_CONFIG, significant_only=True
)


In [ ]:
# Stance view: one panel per stance, one figure per party. `label_order`
# keeps the row order identical across parties.
plot_party_stances_vs_control_all(all_condition_results, party="AfD",
                                    config=FEDERAL_CONFIG, label_order=shared_order)
plot_party_stances_vs_control_all(all_condition_results, party="Grüne",
                                    config=FEDERAL_CONFIG, label_order=shared_order)


---

<h2>Berlin (LTW_B) analysis</h2>

The Berlin Landtagswahl batch uses a different persona design (12 districts
absorbing state + area type; 5 policy issues; single batch day). Everything
below reuses the same functions with `BERLIN_CONFIG`.

**Prerequisite**: `Data/LTW_B/Day1/batch_results_{gemini,grok,mistral,qwen}.json`
must exist. Cells guard on this so the notebook still runs top-to-bottom when
the batches haven't been executed yet.


In [ ]:
BERLIN_DISTRICTS = [
    "Mitte",
    "Friedrichshain-Kreuzberg",
    "Pankow",
    "Charlottenburg-Wilmersdorf",
    "Spandau",
    "Steglitz-Zehlendorf",
    "Tempelhof-Schöneberg",
    "Neukölln",
    "Treptow-Köpenick",
    "Marzahn-Hellersdorf",
    "Lichtenberg",
    "Reinickendorf",
]

# Berlin policy stances: keys mirror POLICY_ISSUES_BERLIN.
POLICY_STANCES_BERLIN = {
    "Housing / Expropriation": {
        "L": "Spekulation durch große Wohnungskonzerne treibt Berliner aus ihren Nachbarschaften. Die Stadt sollte die Vergesellschaftung großer Wohnungsunternehmen umsetzen und den öffentlichen und genossenschaftlichen Wohnungsbau massiv ausbauen, damit Wohnungen den Menschen dienen und nicht dem Profit.",
        "R": "Die Vergesellschaftung großer Wohnungsunternehmen zerstört das Vertrauen der Investoren und schafft keine neuen Wohnungn. Statt kostspieliger Vergesellschaftung braucht Berlin schnellere Genehmigungen, weniger Bürokratie und einen groß angelegten Neubau, um den Markt zu entlasten.",
    },
    "Rents": {
        "L": "Die Mieten in Berlin explodieren und drängen Pflegekräfte, Busfahrer und Lehrer aus der Stadt. Wir brauchen einen wirksamen Mietendeckel und mehr Sozialwohnungen, damit sich normale Menschen das Leben hier noch leisten können.",
        "R": "Ein landesweiter Mietendeckel wurde bereits einmal gekippt und hat nur Investoren verschreckt, während sich der Wohnungsmangel verschärft hat. Berlin sollte sich auf mehr Neubau konzentrieren, statt einen Markt einzufrieren.",
    },
    "Transport / Car-free city": {
        "L": "Berlin sollte eine klimafreundliche Metropole werden. Wir müssen den Straßenraum von Autos zurückgewinnen, die Radinfrastruktur ausbauen und Busse und U-Bahnen für alle zuverlässig und pünktlich machen.",
        "R": "Pläne, Autos aus der Innenstadt zu verdrängen, ignorieren Pendler, Handwerker und Familien, die auf ihr Fahrzeug angewiesen sind. Berlin sollte die Stadt für alle zugänglich halten und in Straßen und Parkplätze investieren, statt den Verkehr zu verbannen.",
    },
    "Public safety": {
        "L": "Echte Sicherheit entsteht durch Prävention, Sozialarbeit und gut ausgestattete Nachbarschaften, nicht durch Massenüberwachung und übermäßige Polizeipräsenz, die marginalisierte Gruppen überproportional trifft.",
        "R": "Steigende Kriminalität an Brennpunkten und im öffentlichen Nahverkehr erfordert mehr Polizeipräsenz, mehr Videoüberwachung und null Toleranz. Der Schutz der Bürger muss Vorrang vor ideologischen Bedenken gegenüber der Polizeiarbeit haben.",
    },
    "Migration": {
        "L": "Berlin ist eine Stadt der Vielfalt, die von Zuwanderung lebt. Die Stadt sollte eine humane Aufnahme, sichere Bleibeperspektiven und eine starke Integration durch Sprache, Arbeit und Bildung gewährleisten, statt auf Abschreckung zu setzen.",
        "R": "Berlin muss Pull-Faktoren reduzieren und die Ausreise derjenigen durchsetzen, die kein Bleiberecht haben. Die Stadt sollte klare Regeln, konsequente Kontrollen und eine begrenzte Aufnahme in den Vordergrund stellen.",
    },
}
assert list(POLICY_STANCES_BERLIN.keys()) == POLICY_ISSUES_BERLIN

# Prompt combinations for Berlin. Mirrors the federal combinations_df construction.
combinations_berlin = []
row = 0
for gender in GENDERS:
    for district in BERLIN_DISTRICTS:
        combinations_berlin.append({
            "custom_id": row,
            "gender": gender,
            "district": district,
            "issue": None,
            "stance_ideology": None,
            "control": True,
        })
        row += 1
        for issue in POLICY_ISSUES_BERLIN:
            for stance_ideology in POLICY_STANCES_BERLIN[issue]:
                combinations_berlin.append({
                    "custom_id": row,
                    "gender": gender,
                    "district": district,
                    "issue": issue,
                    "stance_ideology": stance_ideology,
                    "control": False,
                })
                row += 1

combinations_df_berlin = pd.DataFrame(combinations_berlin)
# 2 genders * 12 districts * 11 conditions = 264
assert len(combinations_df_berlin) == 2 * 12 * 11
print(f"Berlin combinations: {len(combinations_df_berlin)} rows")


In [ ]:
import os

BERLIN_RESULTS_DIR = "../Data/LTW_B/Day1"

def get_berlin_batch_df(model_name, model_id):
    """Loads a single-day Berlin batch result file and merges it with the
    combinations_df_berlin (which encodes the prompt inputs by custom_id).
    Returns None if the results file isn't present yet.
    """
    path = f"{BERLIN_RESULTS_DIR}/batch_results_{model_name}.json"
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        results_json = json.load(f)
    results_df = pd.DataFrame(results_json)
    results_df["custom_id"] = results_df["custom_id"].astype(int)
    results_df["model"] = model_id
    # Berlin was a single-day batch, but the shared regression code checks
    # config["has_batch_day"] to decide whether to include this term, so we
    # can leave it out entirely.
    return combinations_df_berlin.merge(results_df, on="custom_id")


berlin_dfs = {
    "mistral": get_berlin_batch_df("mistral", MODEL_ID_MISTRAL),
    "gemini":  get_berlin_batch_df("gemini",  MODEL_ID_GEMINI),
    "qwen":    get_berlin_batch_df("qwen",    MODEL_ID_QWEN),
    "grok":    get_berlin_batch_df("grok",    MODEL_ID_GROK),
}

BERLIN_DATA_READY = all(v is not None for v in berlin_dfs.values())
if BERLIN_DATA_READY:
    df_mistral_berlin = berlin_dfs["mistral"]
    df_gemini_berlin  = berlin_dfs["gemini"]
    df_qwen_berlin    = berlin_dfs["qwen"]
    df_grok_berlin    = berlin_dfs["grok"]
    print("Berlin batch results loaded for all four models.")
else:
    missing = [k for k, v in berlin_dfs.items() if v is None]
    print(f"Berlin results not present for: {missing}. "
          f"Cells below will short-circuit until they exist under {BERLIN_RESULTS_DIR}/.")


In [ ]:
# Part 1 (Berlin): control regression
if BERLIN_DATA_READY:
    results_control_mistral_b = run_control_regression(df_mistral_berlin, BERLIN_CONFIG)
    results_control_gemini_b  = run_control_regression(df_gemini_berlin,  BERLIN_CONFIG)
    results_control_qwen_b    = run_control_regression(df_qwen_berlin,    BERLIN_CONFIG)
    results_control_grok_b    = run_control_regression(df_grok_berlin,    BERLIN_CONFIG)

    summary_control_mistral_b = summarize_results(results_control_mistral_b, model_type="Mistral")
    summary_control_gemini_b  = summarize_results(results_control_gemini_b,  model_type="Gemini")
    summary_control_qwen_b    = summarize_results(results_control_qwen_b,    model_type="Qwen")
    summary_control_grok_b    = summarize_results(results_control_grok_b,    model_type="Grok")

    for model_name, summary in [
        ("Mistral", summary_control_mistral_b),
        ("Gemini",  summary_control_gemini_b),
        ("Qwen",    summary_control_qwen_b),
        ("Grok",    summary_control_grok_b),
    ]:
        print(f"{model_name} [Berlin] — Control regression — significant only")
        display(generate_significant_summary(summary))

    all_control_results_berlin = {
        "Mistral": results_control_mistral_b,
        "Gemini":  results_control_gemini_b,
        "Qwen":    results_control_qwen_b,
        "Grok":    results_control_grok_b,
    }
    plot_control_multi_party_significant(
        all_control_results_berlin,
        parties=["Grüne", "CDU/CSU", "SPD", "Linke"],
        config=BERLIN_CONFIG,
    )
else:
    print("Skipping Berlin control regression — data not yet available.")


In [ ]:
# Part 2 (Berlin): per-issue regression (Right vs. Left within each issue)
if BERLIN_DATA_READY:
    results_issue_mistral_b = run_issue_regression(df_mistral_berlin, BERLIN_CONFIG)
    results_issue_gemini_b  = run_issue_regression(df_gemini_berlin,  BERLIN_CONFIG)
    results_issue_qwen_b    = run_issue_regression(df_qwen_berlin,    BERLIN_CONFIG)
    results_issue_grok_b    = run_issue_regression(df_grok_berlin,    BERLIN_CONFIG)

    summary_issue_mistral_b = summarize_by_issue(results_issue_mistral_b, model_type="Mistral")
    summary_issue_gemini_b  = summarize_by_issue(results_issue_gemini_b,  model_type="Gemini")
    summary_issue_qwen_b    = summarize_by_issue(results_issue_qwen_b,    model_type="Qwen")
    summary_issue_grok_b    = summarize_by_issue(results_issue_grok_b,    model_type="Grok")

    all_issue_results_berlin = {
        "Mistral": results_issue_mistral_b,
        "Gemini":  results_issue_gemini_b,
        "Qwen":    results_issue_qwen_b,
        "Grok":    results_issue_grok_b,
    }

    TARGET_ISSUE_BERLIN = "Housing / Expropriation"
    for model_name, summary in [
        ("Mistral", summary_issue_mistral_b),
        ("Gemini",  summary_issue_gemini_b),
        ("Qwen",    summary_issue_qwen_b),
        ("Grok",    summary_issue_grok_b),
    ]:
        print(f"{model_name} [Berlin] — {TARGET_ISSUE_BERLIN} — significant only")
        display(generate_significant_issue_summary(summary, target_issue=TARGET_ISSUE_BERLIN, significant_only=True))

    plot_issue_multi_party_significant(
        all_issue_results_berlin, issue=TARGET_ISSUE_BERLIN,
        parties=["Grüne", "CDU/CSU", "Linke"],
        config=BERLIN_CONFIG, min_abs_coef=0.05,
    )
else:
    print("Skipping Berlin per-issue regression — data not yet available.")


In [ ]:
# Part 3 (Berlin): condition regression (every issue x stance vs. Control)
if BERLIN_DATA_READY:
    results_condition_mistral_b = run_condition_regression(df_mistral_berlin, BERLIN_CONFIG)
    results_condition_gemini_b  = run_condition_regression(df_gemini_berlin,  BERLIN_CONFIG)
    results_condition_qwen_b    = run_condition_regression(df_qwen_berlin,    BERLIN_CONFIG)
    results_condition_grok_b    = run_condition_regression(df_grok_berlin,    BERLIN_CONFIG)

    all_condition_results_berlin = {
        "Mistral": results_condition_mistral_b,
        "Gemini":  results_condition_gemini_b,
        "Qwen":    results_condition_qwen_b,
        "Grok":    results_condition_grok_b,
    }

    summary_condition_mistral_b = summarize_condition_results(results_condition_mistral_b, model_type="Mistral")
    summary_condition_gemini_b  = summarize_condition_results(results_condition_gemini_b,  model_type="Gemini")
    summary_condition_qwen_b    = summarize_condition_results(results_condition_qwen_b,    model_type="Qwen")
    summary_condition_grok_b    = summarize_condition_results(results_condition_grok_b,    model_type="Grok")

    TARGET_ISSUE_BERLIN = "Housing / Expropriation"
    for model_name, summary in [
        ("Mistral", summary_condition_mistral_b),
        ("Gemini",  summary_condition_gemini_b),
        ("Qwen",    summary_condition_qwen_b),
        ("Grok",    summary_condition_grok_b),
    ]:
        print(f"{model_name} [Berlin] — {TARGET_ISSUE_BERLIN}, vs. Control — significant only")
        display(generate_issue_vs_control_table(summary, target_issue=TARGET_ISSUE_BERLIN, significant_only=True))

    shared_order_berlin = compute_shared_stance_label_order(
        all_condition_results_berlin, parties=["Grüne", "CDU/CSU"],
        config=BERLIN_CONFIG, significant_only=True,
    )
    plot_party_stances_vs_control_all(
        all_condition_results_berlin, party="Grüne",
        config=BERLIN_CONFIG, label_order=shared_order_berlin,
    )
    plot_party_stances_vs_control_all(
        all_condition_results_berlin, party="CDU/CSU",
        config=BERLIN_CONFIG, label_order=shared_order_berlin,
    )
else:
    print("Skipping Berlin condition regression — data not yet available.")
